# Polymarket Cold-Start Trade Edge

**Can market and context features identify positive-alpha trades for wallets never seen in training?**

The main test is a **future, wallet-disjoint holdout**: the final evaluation set contains only wallets that never appear in training.

- **Prediction target:** `outcome_correct`
- **Economic score:** `realized_edge = outcome_correct - avg_price`
- **Ranking score:** `predicted_alpha = model_prob - avg_price`

All logic lives in `src/machine_learning/`; this notebook is a thin driver that reproduces the full pipeline end-to-end.

## 0. Setup

In [1]:
# Anchor working directory to the project root so paths resolve consistently
import os
from pathlib import Path

_p = Path.cwd()
while not (_p / 'src').exists() and _p != _p.parent:
    _p = _p.parent
os.chdir(_p)
print('Working dir:', os.getcwd())

Working dir: /home/john-stewart/Documents/vault/UCD/data_sci_fi/02-project/polymarket_insider/polymarket-whale-watch


In [2]:
from src.machine_learning.features.build import build_feature_matrix
from src.machine_learning.split import make_cold_start_split
from src.machine_learning.train import (
    train_baselines, train_main_model, train_feature_ablation,
)
from src.machine_learning.evaluate import (
    warm_vs_cold, economic_ranking, robustness, bootstrap_uplift,
    walkforward, feature_importance,
)
from src.machine_learning.deployment import (
    wallet_scope_audit, friction_analysis, risk_metrics, aum_scaling_table,
)
from src.machine_learning.plots import (
    plot_eda_diagnostics, plot_evaluation_grid,
    plot_feature_importance, plot_risk_profile,
)

## 1 .Build the Feature Matrix

Runs the full feature pipeline: DuckDB query → collapse fills to positions → wallet binomial-edge test → point-in-time history features → sentiment joins (Truth Social + FT) → divergence scores and transforms. Cached to parquet; set `use_cache=False` to force a rebuild.

In [ ]:
fm, cutoff = build_feature_matrix()
fm.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Dataset Diagnostics

Four views of the feature matrix: market-price calibration, realized-edge distribution, edge by time-to-resolution, edge by position size.

In [ ]:
plot_eda_diagnostics(fm)

## 3. Train / Cold / Warm Split

Wallet-disjoint holdout: `cold_test` contains only wallets that never appear in `train`. `warm_test` is the late-period fallback for comparison.

In [ ]:
train, cold_test, warm_test, train_pool, late_pool = make_cold_start_split(fm, cutoff)

y_train = train['outcome_correct'].astype(int)
y_test = cold_test['outcome_correct'].astype(int)
market_probs = cold_test['market_implied_prob'].to_numpy()
market_probs_warm = warm_test['market_implied_prob'].to_numpy()

In [ ]:
scope_df, overlap = wallet_scope_audit(train, cold_test, warm_test)
print(f'Wallet overlap between train and cold test: {overlap}')
scope_df

## 4. Baselines

Market-price-only and logistic regression as reference points. Anything the main model does has to beat these.

In [ ]:
baseline_df, logit_probs = train_baselines(train, cold_test, y_train, y_test, market_probs)
baseline_df.round(4)

## 5. Main XGBoost Model

Wallet-grouped CV + Optuna tuning on the market+context feature set. The tuned model is selected by CV and then fit on the full training set. Fits a divergence-only variant and a full-feature variant for comparison.

In [ ]:
result = train_main_model(train, cold_test, warm_test, y_train)
result['cv_comparison'].round(4)

## 6. Feature-Family Ablation

Train one model per feature family using the tuned hyperparameters. Tests whether adding history or the full feature set improves on context-only.

In [ ]:
ablation_df = train_feature_ablation(
    train, cold_test, y_train, y_test, market_probs, result['tuned_params']
)
ablation_df.round(4)

## 7. Warm vs Cold Start

Does the model beat market pricing even on wallets it has never seen? If cold-start metrics collapse to market-only, the result is a wallet-persistence artefact.

In [ ]:
wc_df = warm_vs_cold(
    cold_test, warm_test,
    result['context_probs'], result['context_probs_warm'],
    market_probs, market_probs_warm,
)
wc_df.round(4)

## 8. Economic Ranking Quality

The model is only useful if the top-alpha trades generate positive realized edge. Compares main-model alpha against divergence-only and market-price baselines, then shows the edge curve across different selection fractions.

In [ ]:
selection_df, alpha_curve = economic_ranking(
    cold_test, result['context_probs'], result['div_probs'], market_probs,
)
selection_df.round(4)

In [ ]:
alpha_curve.round(4)

## 9 . Robustness Across Holdouts

Repeat the cold-start holdout with different random wallet samples. If metrics hold up across seeds, the result isn't an artefact of one favourable split.

In [ ]:
robust_df, robust_summary = robustness(train_pool, late_pool, result['tuned_params'])
robust_df.round(4)

In [ ]:
robust_summary.round(4)

## 10. Evaluation Grid

In [ ]:
plot_evaluation_grid(
    y_test, market_probs, logit_probs,
    result['baseline_context_probs'], result['context_probs'],
    alpha_curve, selection_df, robust_df,
)

## 11. Feature Importance


In [ ]:
plot_feature_importance(result['final_model'])

## 12. Wallet-Clustered Bootstrap Uplift

Resample wallets (not rows) with replacement to get 95% CIs on each metric uplift over the market baseline. Clustering at wallet level is the right inference procedure when trades within a wallet share common information.

In [ ]:
uplift_df = bootstrap_uplift(cold_test, y_test, market_probs, result['context_probs'])
uplift_df.round(4)

## 13. Walk-Forward Evaluation

Stricter than the resolution-date split: each calendar month in the late period is scored using only entry-date-earlier history. Row-weighted summary avoids letting tiny months dominate.

In [ ]:
walk_df, walk_summary = walkforward(fm, result['tuned_params'])
walk_df.round(4)

In [ ]:
walk_summary.round(4)

## 14. Friction Analysis

Top-10% alpha slice with execution filters: min market volume, position cap, market-volume participation cap. Sweeps transaction costs to show P&L sensitivity.

In [ ]:
friction_df, friction_coverage = friction_analysis(
    cold_test, result['context_probs'], market_probs,
)
friction_df.round(4)

In [ ]:
friction_coverage.round(4)

## 15. Risk-Adjusted Performance

In [ ]:
risk_summary, risk_series = risk_metrics(
    cold_test, result['context_probs'], market_probs,
)
risk_summary.round(4)

In [ ]:
plot_risk_profile(risk_series)

## 16. AUM Scaling

Projects annualised P&L across three AUM tiers with liquidity-adjusted scale factors. Team cost of four staff at €120k all-in is compared to net P&L.

In [ ]:
aum_header, aum_tiers = aum_scaling_table()
aum_header

In [ ]:
aum_tiers

## Summary

This pipeline reproduces the full cold-start analysis from `src/machine_learning/`:

- feature engineering (`features/*`),
- wallet-disjoint splitting (`split.py`),
- baseline + tuned XGBoost training (`train.py`),
- cold/warm comparison, economic ranking, robustness, bootstrap uplift, walk-forward (`evaluate.py`),
- friction analysis, risk metrics, AUM scaling (`deployment.py`),
- EDA, evaluation, importance, and risk charts (`plots.py`).

All configuration lives in `src/machine_learning/config.py` — holdout thresholds, Optuna trials, friction parameters, and AUM tier assumptions.